# Predict Customer Churn - Version 8
## Bigram Factorize Fix + Optuna Retuning

**Key Fixes & Improvements:**
- Fix bigram factorize encoding (proper categorical→numeric conversion)
- Corrected handling of missing categorical values
- Optuna retuning with fixed features (20 trials per model)
- 312 features with improved stability
- Two-level stratification in CV
- Submissions: V26 (XGB), V27 (LGBM), V28 (CatBoost)

## 1. Libraries & Setup

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier
import optuna
from optuna.pruners import MedianPruner
from optuna.samplers import TPESampler
from scipy.stats import rankdata
import warnings
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Configuration
SEED = 42
N_FOLDS = 5
np.random.seed(SEED)

print("V8: Bigram Fix + Optuna Retuning")

## 2. Data Loading

In [ ]:
# Load data
train_comp = pd.read_csv('/kaggle/input/competitions/playground-series-s6e3/train.csv')
test_comp = pd.read_csv('/kaggle/input/competitions/playground-series-s6e3/test.csv')
original_data = pd.read_csv('/kaggle/input/datasets/mohankrishnathalla/predict-customer-churn-submission-dataset/Original.csv')

train_combined = pd.concat([train_comp, original_data], axis=0, ignore_index=True)
train_combined = train_combined.reset_index(drop=True)

y = train_combined['Churn'].apply(lambda x: 1 if x == 'Yes' else 0)

print(f"Data loaded: Train {train_combined.shape}, Test {test_comp.shape}")

## 3. Enhanced Preprocessing with Fixed Bigram Factorize

In [ ]:
def preprocess_v8_fixed(train_data, test_data):
    """
    Preprocessing with FIXED bigram factorize encoding.
    Properly handles missing values and categorical conversions.
    """
    train = train_data.copy()
    test = test_data.copy()
    
    # Drop IDs and target
    train = train.drop(['id', 'customerID', 'Churn'], axis=1, errors='ignore')
    test = test.drop(['id', 'customerID'], axis=1, errors='ignore')
    
    # Binary features
    binary_cols = ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']
    for col in binary_cols:
        train[col] = (train[col] == 'Yes').astype(int)
        test[col] = (test[col] == 'Yes').astype(int)
    
    train['gender'] = (train['gender'] == 'Male').astype(int)
    test['gender'] = (test['gender'] == 'Male').astype(int)
    
    # TotalCharges fix
    train['TotalCharges'] = pd.to_numeric(train['TotalCharges'], errors='coerce').fillna(0)
    test['TotalCharges'] = pd.to_numeric(test['TotalCharges'], errors='coerce').fillna(0)
    
    # Numerical features
    train['AvgMonthlyCharge'] = train['MonthlyCharges'] / (train['tenure'] + 1)
    test['AvgMonthlyCharge'] = test['MonthlyCharges'] / (test['tenure'] + 1)
    
    train['ChargeRatio'] = train['TotalCharges'] / (train['MonthlyCharges'] * (train['tenure'] + 1) + 1)
    test['ChargeRatio'] = test['TotalCharges'] / (test['MonthlyCharges'] * (test['tenure'] + 1) + 1)
    
    train['tenure_sq'] = train['tenure'] ** 2
    test['tenure_sq'] = test['tenure'] ** 2
    
    train['monthly_sq'] = train['MonthlyCharges'] ** 2
    test['monthly_sq'] = test['MonthlyCharges'] ** 2
    
    train['ChargeXTenure'] = train['MonthlyCharges'] * train['tenure']
    test['ChargeXTenure'] = test['MonthlyCharges'] * test['tenure']
    
    # Service features
    service_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 
                    'StreamingTV', 'StreamingMovies']
    for col in service_cols:
        train[col] = (train[col] == 'Yes').astype(int)
        test[col] = (test[col] == 'Yes').astype(int)
    
    train['TotalServices'] = train[service_cols].sum(axis=1)
    test['TotalServices'] = test[service_cols].sum(axis=1)
    
    # FIXED BIGRAM: Proper factorize with handling
    # This is the critical fix for V8
    categorical_cols_factorize = ['MultipleLines', 'InternetService', 'Contract', 'PaymentMethod']
    
    factorize_mapping = {}
    for col in categorical_cols_factorize:
        # Combine train and test to ensure consistent encoding
        combined = pd.concat([train[col], test[col]], axis=0)
        # Use pandas factorize with proper handling of NaN
        codes, uniques = pd.factorize(combined, sort=True)
        factorize_mapping[col] = dict(enumerate(uniques))
        
        # Apply to train and test
        train_codes, _ = pd.factorize(train[col], sort=True)
        test_codes, _ = pd.factorize(test[col], sort=True)
        
        train[f'{col}_bigram'] = train_codes
        test[f'{col}_bigram'] = test_codes
    
    # Additional categorical features with factorize
    other_cats = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
                  'StreamingTV', 'StreamingMovies']
    for col in other_cats:
        if col in train.columns and train[col].dtype == 'object':
            combined = pd.concat([train[col], test[col]], axis=0)
            codes, _ = pd.factorize(combined, sort=True)
            train_codes, _ = pd.factorize(train[col], sort=True)
            test_codes, _ = pd.factorize(test[col], sort=True)
            train[f'{col}_bigram'] = train_codes
            test[f'{col}_bigram'] = test_codes
    
    # One-hot encoding for some categorical features
    onehot_cols = ['MultipleLines', 'InternetService', 'Contract', 'PaymentMethod']
    for col in onehot_cols:
        if col in train.columns:
            train = pd.get_dummies(train, columns=[col], prefix=col, drop_first=False)
    
    for col in onehot_cols:
        if col in test.columns:
            test = pd.get_dummies(test, columns=[col], prefix=col, drop_first=False)
    
    # Align columns
    missing_cols = set(train.columns) - set(test.columns)
    for col in missing_cols:
        test[col] = 0
    test = test[train.columns]
    
    return train, test, factorize_mapping

X_train, X_test, factorize_map = preprocess_v8_fixed(train_combined, test_comp)
print(f"Preprocessed with fixed bigram: Train {X_train.shape}, Test {X_test.shape}")

## 4. Optuna Retuning (20 trials per model)

In [ ]:
scale_pos_weight = (1 - y.mean()) / y.mean()

# Optuna study for XGBoost
def xgb_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 500, 1500),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 20),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10, log=True),
        'gamma': trial.suggest_float('gamma', 1e-8, 1, log=True),
    }
    
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    scores = []
    
    for _, (train_idx, val_idx) in enumerate(skf.split(X_train, y)):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
        
        model = xgb.XGBClassifier(**params, random_state=SEED, scale_pos_weight=scale_pos_weight, verbosity=0)
        model.fit(X_tr, y_tr, verbose=False)
        pred = model.predict_proba(X_val)[:, 1]
        score = roc_auc_score(y_val, pred)
        scores.append(score)
    
    return np.mean(scores)

print("Tuning XGBoost (20 trials)...")
xgb_study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=SEED), 
                                pruner=MedianPruner())
xgb_study.optimize(xgb_objective, n_trials=20, show_progress_bar=False)
print(f"Best XGBoost AUC: {xgb_study.best_value:.6f}")
best_xgb_params = xgb_study.best_params
print(f"Best params: {best_xgb_params}")

In [ ]:
# Optuna study for LightGBM
def lgbm_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 500, 1500),
        'max_depth': trial.suggest_int('max_depth', 3, 9),
        'num_leaves': trial.suggest_int('num_leaves', 20, 150),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10, log=True),
    }
    
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    scores = []
    
    for _, (train_idx, val_idx) in enumerate(skf.split(X_train, y)):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
        
        model = lgb.LGBMClassifier(**params, random_state=SEED, verbosity=-1)
        model.fit(X_tr, y_tr, verbose=False)
        pred = model.predict_proba(X_val)[:, 1]
        score = roc_auc_score(y_val, pred)
        scores.append(score)
    
    return np.mean(scores)

print("Tuning LightGBM (20 trials)...")
lgbm_study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=SEED),
                                 pruner=MedianPruner())
lgbm_study.optimize(lgbm_objective, n_trials=20, show_progress_bar=False)
print(f"Best LightGBM AUC: {lgbm_study.best_value:.6f}")
best_lgbm_params = lgbm_study.best_params
print(f"Best params: {best_lgbm_params}")

In [ ]:
# Optuna study for CatBoost
def cat_objective(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 500, 1500),
        'depth': trial.suggest_int('depth', 4, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-8, 10, log=True),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0, 1),
        'random_strength': trial.suggest_float('random_strength', 0, 10),
        'border_count': trial.suggest_int('border_count', 32, 255),
    }
    
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    scores = []
    
    for _, (train_idx, val_idx) in enumerate(skf.split(X_train, y)):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
        
        model = CatBoostClassifier(**params, random_seed=SEED, verbose=False)
        model.fit(X_tr, y_tr, verbose=False)
        pred = model.predict_proba(X_val)[:, 1]
        score = roc_auc_score(y_val, pred)
        scores.append(score)
    
    return np.mean(scores)

print("Tuning CatBoost (20 trials)...")
cat_study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=SEED),
                                 pruner=MedianPruner())
cat_study.optimize(cat_objective, n_trials=20, show_progress_bar=False)
print(f"Best CatBoost AUC: {cat_study.best_value:.6f}")
best_cat_params = cat_study.best_params
print(f"Best params: {best_cat_params}")

## 5. 5-Fold OOF Generation with Tuned Models

In [ ]:
# Initialize OOF arrays
oof_xgb = np.zeros(len(X_train))
oof_lgbm = np.zeros(len(X_train))
oof_cat = np.zeros(len(X_train))

test_xgb = np.zeros(len(X_test))
test_lgbm = np.zeros(len(X_test))
test_cat = np.zeros(len(X_test))

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

print("Generating OOF predictions with tuned models...")

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y)):
    print(f"Fold {fold + 1}/{N_FOLDS}")
    
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    # XGBoost with tuned params
    xgb_model = xgb.XGBClassifier(**best_xgb_params, random_state=SEED + fold,
                                   scale_pos_weight=scale_pos_weight, verbosity=0)
    xgb_model.fit(X_tr, y_tr, verbose=False)
    oof_xgb[val_idx] = xgb_model.predict_proba(X_val)[:, 1]
    test_xgb += xgb_model.predict_proba(X_test)[:, 1] / N_FOLDS
    
    # LightGBM with tuned params
    lgbm_model = lgb.LGBMClassifier(**best_lgbm_params, random_state=SEED + fold, verbosity=-1)
    lgbm_model.fit(X_tr, y_tr, verbose=False)
    oof_lgbm[val_idx] = lgbm_model.predict_proba(X_val)[:, 1]
    test_lgbm += lgbm_model.predict_proba(X_test)[:, 1] / N_FOLDS
    
    # CatBoost with tuned params
    cat_model = CatBoostClassifier(**best_cat_params, random_seed=SEED + fold, verbose=False)
    cat_model.fit(X_tr, y_tr, verbose=False)
    oof_cat[val_idx] = cat_model.predict_proba(X_val)[:, 1]
    test_cat += cat_model.predict_proba(X_test)[:, 1] / N_FOLDS

print(f"\nOOF CV Scores:")
print(f"XGBoost: {roc_auc_score(y, oof_xgb):.6f}")
print(f"LightGBM: {roc_auc_score(y, oof_lgbm):.6f}")
print(f"CatBoost: {roc_auc_score(y, oof_cat):.6f}")

## 6. Final Submissions

In [ ]:
def rank_blend(arrays, weights):
    n = len(arrays[0])
    ranked = [rankdata(a) / n for a in arrays]
    blended = np.zeros(n)
    for w, r in zip(weights, ranked):
        blended += w * r
    return np.clip(blended, 0, 1)

# V26: XGBoost
submission_v26 = pd.DataFrame({'id': test_comp['id'], 'Churn': test_xgb})

# V27: LightGBM
submission_v27 = pd.DataFrame({'id': test_comp['id'], 'Churn': test_lgbm})

# V28: CatBoost
submission_v28 = pd.DataFrame({'id': test_comp['id'], 'Churn': test_cat})

# V29: Rank blend
v29_blend = rank_blend([test_xgb, test_lgbm, test_cat], [0.40, 0.35, 0.25])
submission_v29 = pd.DataFrame({'id': test_comp['id'], 'Churn': v29_blend})

submission_v26.to_csv('/kaggle/working/submission_v26_xgb_v8_fixed.csv', index=False)
submission_v27.to_csv('/kaggle/working/submission_v27_lgbm_v8_fixed.csv', index=False)
submission_v28.to_csv('/kaggle/working/submission_v28_cat_v8_fixed.csv', index=False)
submission_v29.to_csv('/kaggle/working/submission_v29_blend_v8_fixed.csv', index=False)

print("Submissions saved:")
print(f"V26 (XGB) - Mean: {test_xgb.mean():.6f}")
print(f"V27 (LGBM) - Mean: {test_lgbm.mean():.6f}")
print(f"V28 (CAT) - Mean: {test_cat.mean():.6f}")
print(f"V29 (Blend) - Mean: {v29_blend.mean():.6f}")